In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import os

# Set default theme
pio.templates.default = "plotly_dark"

# Define base path
BASE_PATH = "../data/processed"

# Load all processed files
deaths_df = pd.read_csv(os.path.join(BASE_PATH, "total_accidental_deaths_2015_2022.csv"))
transport_df = pd.read_csv(os.path.join(BASE_PATH, "mode_of_transport_2015_2022.csv"))
month_df = pd.read_csv(os.path.join(BASE_PATH, "month_of_occurrence_2015_2022.csv"))
time_df = pd.read_csv(os.path.join(BASE_PATH, "time_of_occurrence_2015_2022.csv"))
road_df = pd.read_csv(os.path.join(BASE_PATH, "road_classification_2015_2022.csv"))
place_df = pd.read_csv(os.path.join(BASE_PATH, "place_of_occurrence_2015_2022.csv"))

print("All files loaded successfully!")
print(f"Deaths: {deaths_df.shape}")
print(f"Transport: {transport_df.shape}")
print(f"Month: {month_df.shape}")
print(f"Time: {time_df.shape}")
print(f"Road: {road_df.shape}")
print(f"Place: {place_df.shape}")

In [ ]:
# Visualization 1 - FINAL PERFECT - Clock text fixed
fig = go.Figure()

# LEFT - Danger Clock
fig.add_trace(go.Barpolar(
    r=time_totals,
    theta=angles,
    width=[45] * 8,
    marker_color=clock_colors,
    marker_line_color="white",
    marker_line_width=1,
    opacity=0.9,
    text=[f"{label}<br>{int(val/1000)}K accidents"
          for label, val in zip(time_labels_clean, time_totals)],
    hovertemplate="%{text}<extra></extra>",
    subplot="polar",
    showlegend=False
))

# RIGHT - Risk zone backgrounds
for y0, y1, color, opacity in [
    (0, 40, "#2a9d8f", 0.08),
    (40, 70, "#e9c46a", 0.08),
    (70, 90, "#f4a261", 0.08),
    (90, 115, "#e63946", 0.15)
]:
    fig.add_shape(
        type="rect",
        x0=time_labels_clean[0], x1=time_labels_clean[-1],
        y0=y0, y1=y1,
        fillcolor=color, opacity=opacity, line_width=0,
        xref="x", yref="y"
    )

# Fill area
fig.add_trace(go.Scatter(
    x=time_labels_clean,
    y=risk_normalized,
    mode="lines",
    line=dict(color="rgba(0,0,0,0)"),
    fill="tozeroy",
    fillcolor="rgba(230,57,70,0.25)",
    hoverinfo="skip",
    showlegend=False,
    xaxis="x", yaxis="y"
))

# Main line
fig.add_trace(go.Scatter(
    x=time_labels_clean,
    y=risk_normalized,
    mode="lines+markers",
    line=dict(color="#e63946", width=4, shape="spline"),
    marker=dict(size=20, color=colors, line=dict(color="white", width=2)),
    customdata=[[l, int(v)] for l, v in zip(time_labels_clean, time_totals)],
    hovertemplate="<b>%{customdata[0]}</b><br>Accidents: %{customdata[1]:,}<extra></extra>",
    showlegend=False,
    xaxis="x", yaxis="y"
))

# Emoji annotations
for i, (label, val, emoji) in enumerate(zip(time_labels_clean, risk_normalized, emojis)):
    fig.add_annotation(
        x=label, y=val + 8,
        text=emoji,
        showarrow=False,
        font=dict(size=20),
        yanchor="bottom",
        xref="x", yref="y"
    )

# Dark Hour annotation
fig.add_annotation(
    x="6PM-9PM", y=110,
    text="<b>⚠️ DARK HOUR</b><br>Most Dangerous",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#e63946",
    arrowwidth=2,
    font=dict(size=11, color="#e63946"),
    bgcolor="rgba(13,13,26,0.9)",
    bordercolor="#e63946",
    borderpad=5,
    ay=-45,
    xref="x", yref="y"
)

# Risk labels
for y, label, color in [
    (20, "LOW RISK", "#2a9d8f"),
    (55, "MEDIUM RISK", "#e9c46a"),
    (80, "HIGH RISK", "#f4a261"),
    (102, "EXTREME RISK", "#e63946")
]:
    fig.add_annotation(
        x="9PM-12AM", y=y,
        text=f"<b>{label}</b>",
        showarrow=False,
        font=dict(size=9, color=color),
        xanchor="right",
        bgcolor="rgba(0,0,0,0.4)",
        borderpad=3,
        xref="x", yref="y"
    )

# Subtitles
fig.add_annotation(
    text="<b>🕰️ Danger Clock</b>",
    x=0.16, y=1.04,
    xref="paper", yref="paper",
    showarrow=False,
    font=dict(size=13, color="white")
)
fig.add_annotation(
    text="<b>🚗 Dark Hours Journey</b>",
    x=0.72, y=1.04,
    xref="paper", yref="paper",
    showarrow=False,
    font=dict(size=13, color="white")
)

fig.update_layout(
    title=dict(
        text="<b>🌙 Dark Hours India — When Roads Turn Deadly</b><br>"
             "<sup>Road Accident Risk by Time of Day — India (2015-2022)</sup>",
        x=0.5,
        font=dict(size=18, color="white")
    ),
    polar=dict(
        domain=dict(x=[0, 0.38], y=[0.05, 0.95]),
        radialaxis=dict(
            visible=True,
            showticklabels=False,
            gridcolor="rgba(255,255,255,0.1)"
        ),
        angularaxis=dict(
            tickvals=angles,
            ticktext=[
                "12AM", "3AM", "6AM", "9AM",
                "12PM", "3PM", "6PM", "9PM"
            ],
            direction="clockwise",
            rotation=90,
            tickfont=dict(size=10, color="white"),
            gridcolor="rgba(255,255,255,0.1)"
        ),
        bgcolor="rgba(0,0,0,0)"
    ),
    xaxis=dict(
        domain=[0.43, 1.0],
        tickfont=dict(size=10, color="white"),
        tickangle=20,
        gridcolor="rgba(255,255,255,0.05)",
        title=dict(text="Time of Day", font=dict(color="white", size=11)),
        categoryorder="array",
        categoryarray=time_labels_clean
    ),
    yaxis=dict(
        tickfont=dict(size=9, color="white"),
        gridcolor="rgba(255,255,255,0.05)",
        title=dict(text="Risk Score (Normalized)", font=dict(color="white", size=11)),
        range=[0, 125]
    ),
    paper_bgcolor="#0d0d1a",
    plot_bgcolor="#0d0d1a",
    font=dict(color="white"),
    showlegend=False,
    height=650,
    width=1400,
    margin=dict(t=130, b=80, l=80, r=120)
)

fig.show()
print("Final visualization rendered!")

In [ ]:
# Visualization 2 - Road Death Fingerprint - Parallel Coordinates FIXED
fig = px.parallel_coordinates(
    state_profile_norm,
    dimensions=metrics,
    labels={
        "Avg Deaths": "Total Deaths",
        "Two Wheeler Deaths": "Two Wheeler",
        "Night %": "Night Risk",
        "Monsoon %": "Monsoon Risk",
        "NH %": "NH Risk",
        "Rural %": "Rural Risk"
    },
    color="Avg Deaths",
    color_continuous_scale=[
        [0, "#1a1a4e"],
        [0.4, "#457b9d"],
        [0.7, "#e63946"],
        [1, "#ff9f1c"]
    ],
)

fig.update_layout(
    paper_bgcolor="#0d0d1a",
    plot_bgcolor="#0d0d1a",
    font=dict(color="white", size=11),
    title=dict(
        text="<b>\U0001f9ec Road Death Fingerprint</b><br><sup>Risk Profile of Each State/UT — India (2015-2022)</sup>",
        x=0.5,
        font=dict(size=18, color="white")
    ),
    coloraxis_colorbar=dict(
        title=dict(text="Death Risk", font=dict(color="white")),
        tickfont=dict(color="white")
    ),
    height=600,
    width=1100,
    margin=dict(t=120, b=80, l=80, r=80)
)

fig.show()
print("Parallel Coordinates chart rendered!")

In [ ]:
# Visualization 3 - Sunburst - FIXED
month_cols = ["January", "February", "March", "April", "May", "June",
              "July", "August", "September", "October", "November", "December"]

def get_season(month):
    if month in ["December", "January", "February"]:
        return "Winter"
    elif month in ["March", "April", "May"]:
        return "Summer"
    elif month in ["June", "July", "August", "September"]:
        return "Monsoon"
    else:
        return "Post Monsoon"

month_totals = month_df[month_cols].sum().reset_index()
month_totals.columns = ["Month", "Total Accidents"]
month_totals["Season"] = month_totals["Month"].apply(get_season)

sunburst_data = []
sunburst_data.append({
    "id": "India", "parent": "", 
    "label": "India",
    "value": int(month_totals["Total Accidents"].sum()),
    "color": "#ffffff"
})

season_order = ["Winter", "Summer", "Monsoon", "Post Monsoon"]
season_colors_map = {
    "Winter": "#4e9af1",
    "Summer": "#f4a261",
    "Monsoon": "#2a9d8f",
    "Post Monsoon": "#e9c46a"
}

for season in season_order:
    season_total = int(month_totals[
        month_totals["Season"] == season]["Total Accidents"].sum())
    sunburst_data.append({
        "id": season, "parent": "India",
        "label": f"{season}",
        "value": season_total,
        "color": season_colors_map[season]
    })

for _, row in month_totals.iterrows():
    sunburst_data.append({
        "id": row["Month"], "parent": row["Season"],
        "label": row["Month"][:3],
        "value": int(row["Total Accidents"]),
        "color": season_colors_map[row["Season"]]
    })

df_sunburst = pd.DataFrame(sunburst_data)

fig = go.Figure(go.Sunburst(
    ids=df_sunburst["id"],
    labels=df_sunburst["label"],
    parents=df_sunburst["parent"],
    values=df_sunburst["value"],
    marker=dict(
        colors=df_sunburst["color"],
        line=dict(color="#0d0d1a", width=2)
    ),
    branchvalues="total",
    hovertemplate="<b>%{label}</b><br>Accidents: %{value:,}<extra></extra>",
    textfont=dict(size=12, color="white"),
    insidetextorientation="radial"
))

fig.update_layout(
    title=dict(
        text="<b>\U0001f326\ufe0f Accident Weather Map</b><br>"
             "<sup>Season & Month wise Road Accidents — India (2015-2022)</sup>",
        x=0.5,
        font=dict(size=18, color="white")
    ),
    paper_bgcolor="#0d0d1a",
    font=dict(color="white"),
    height=750,
    width=750,
    margin=dict(t=120, b=80, l=20, r=20),
    annotations=[
        dict(
            text="\U0001f7e2 Monsoon most dangerous  |  \U0001f535 January peak month",
            x=0.5, y=-0.08,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=12, color="white"),
            align="center",
            bgcolor="rgba(13,13,26,0.8)",
            borderpad=6
        )
    ]
)

fig.show()
print("Sunburst chart rendered!")

In [ ]:
# Visualization 4 - Hour x Month x State - 3D with Rotation Animation

import numpy as np

month_cols = ["January", "February", "March", "April", "May", "June",
              "July", "August", "September", "October", "November", "December"]

time_cols = [
    "0000 hrs to 0300 hrs (Night)",
    "0300 hrs to 0600 hrs (Night)",
    "0600 hrs to 0900 hrs (Day)",
    "0900 hrs to 1200 hrs (Day)",
    "1200 hrs to 1500 hrs (Day)",
    "1500 hrs to 1800 hrs (Day)",
    "1800 hrs to 2100 hrs (Night)",
    "2100 hrs to 2400 hrs (Night)"
]

time_short = ["12AM-3AM", "3AM-6AM", "6AM-9AM", "9AM-12PM",
              "12PM-3PM", "3PM-6PM", "6PM-9PM", "9PM-12AM"]

month_short = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

all_states = sorted(deaths_df["State/UT"].unique().tolist())

# Build data for each state
state_data = {}
for state in all_states:
    m_data = month_df[month_df["State/UT"] == state][month_cols].sum()
    m_prop = m_data / m_data.sum()
    t_data = time_df[time_df["State/UT"] == state][time_cols].sum()
    t_prop = t_data / t_data.sum()
    total_deaths = deaths_df[deaths_df["State/UT"] == state]["Total Accidental Deaths"].sum()
    z_matrix = []
    for m in m_prop:
        row = [round(m * t * total_deaths) for t in t_prop]
        z_matrix.append(row)
    state_data[state] = z_matrix

first_state = all_states[0]

# Camera rotation frames
frames = []
n_frames = 60
for k in range(n_frames):
    angle = k * (360 / n_frames)
    angle_rad = np.radians(angle)
    eye_x = 2.0 * np.cos(angle_rad)
    eye_y = 2.0 * np.sin(angle_rad)
    eye_z = 1.2
    frames.append(go.Frame(
        layout=dict(
            scene_camera=dict(
                eye=dict(x=eye_x, y=eye_y, z=eye_z)
            )
        ),
        name=str(k)
    ))

fig = go.Figure(frames=frames)

# Add surface traces
for i, state in enumerate(all_states):
    fig.add_trace(go.Surface(
        z=state_data[state],
        x=time_short,
        y=month_short,
        colorscale=[
            [0, "#0d0d1a"],
            [0.3, "#1a1a4e"],
            [0.6, "#e63946"],
            [1, "#ff9f1c"]
        ],
        showscale=(i == 0),
        colorbar=dict(
            title=dict(text="Accidents", font=dict(color="white")),
            tickfont=dict(color="white")
        ),
        name=state,
        visible=(i == 0),
        hovertemplate=(
            "<b>" + state + "</b><br>"
            "Time: %{x}<br>"
            "Month: %{y}<br>"
            "Accidents: %{z:,}<extra></extra>"
        )
    ))

# Dropdown buttons
buttons = []
for i, state in enumerate(all_states):
    visibility = [False] * len(all_states)
    visibility[i] = True
    buttons.append(dict(
        label=state,
        method="update",
        args=[
            {"visible": visibility},
            {"title": {"text": f"<b>\U0001f4ca Hour x Month x Accidents</b><br>"
                       f"<sup>{state} — 2015-2022</sup>"}}
        ]
    ))

fig.update_layout(
    title=dict(
        text=f"<b>\U0001f4ca Hour x Month x Accidents — 3D Surface</b><br>"
             f"<sup>{first_state} — Use dropdown to switch states</sup>",
        x=0.5,
        font=dict(size=16, color="white")
    ),
    updatemenus=[
        # Dropdown
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.02,
            xanchor="left",
            y=1.15,
            yanchor="top",
            bgcolor="#1a1a2e",
            bordercolor="#e63946",
            font=dict(color="white", size=11)
        ),
        # Play/Pause buttons
        dict(
            type="buttons",
            showactive=False,
            x=0.5,
            xanchor="center",
            y=-0.08,
            yanchor="top",
            bgcolor="#1a1a2e",
            bordercolor="#e63946",
            font=dict(color="white", size=11),
            buttons=[
                dict(
                    label="▶ Rotate",
                    method="animate",
                    args=[None, dict(
                        frame=dict(duration=50, redraw=True),
                        fromcurrent=True,
                        transition=dict(duration=0),
                        mode="immediate",
                        loop=True
                    )]
                ),
                dict(
                    label="⏸ Stop",
                    method="animate",
                    args=[[None], dict(
                        frame=dict(duration=0, redraw=False),
                        mode="immediate",
                        transition=dict(duration=0)
                    )]
                )
            ]
        )
    ],
    scene=dict(
        xaxis=dict(
            title=dict(text="Time Slot", font=dict(color="white")),
            tickfont=dict(color="white", size=9),
            gridcolor="rgba(255,255,255,0.1)",
            backgroundcolor="#0d0d1a"
        ),
        yaxis=dict(
            title=dict(text="Month", font=dict(color="white")),
            tickfont=dict(color="white", size=9),
            gridcolor="rgba(255,255,255,0.1)",
            backgroundcolor="#0d0d1a"
        ),
        zaxis=dict(
            title=dict(text="Accidents", font=dict(color="white")),
            tickfont=dict(color="white", size=9),
            gridcolor="rgba(255,255,255,0.1)",
            backgroundcolor="#0d0d1a"
        ),
        bgcolor="#0d0d1a",
        camera=dict(eye=dict(x=2.0, y=2.0, z=1.2))
    ),
    paper_bgcolor="#0d0d1a",
    plot_bgcolor="#0d0d1a",
    font=dict(color="white"),
    height=700,
    width=1100,
    margin=dict(t=150, b=80, l=50, r=50)
)

fig.show()
print("3D Animated visualization rendered!")

In [ ]:
# Visualization 5 - Two Wheeler Heatwave - Smart Text

tw_all = tw_state.sort_values("Two_Wheeler_Deaths", ascending=False).reset_index(drop=True)

colors_all = pc.sample_colorscale(
    [[0, "#e63946"], [0.3, "#ff6b35"], [0.6, "#f4a261"], [0.8, "#457b9d"], [1, "#1a1a4e"]],
    len(tw_all)
)

# Only show text for top 10 — rest empty
total = tw_all["Two_Wheeler_Deaths"].sum()
text_labels = []
for i, row in tw_all.iterrows():
    pct = row["Two_Wheeler_Deaths"] / total * 100
    if pct >= 2.5:  # Only show text if slice is big enough
        text_labels.append(f"<b>{row['State/UT'].split()[0]}</b><br>{pct:.1f}%")
    else:
        text_labels.append("")

fig = go.Figure()

fig.add_trace(go.Pie(
    labels=tw_all["State/UT"],
    values=tw_all["Two_Wheeler_Deaths"],
    hole=0.55,
    marker=dict(
        colors=colors_all,
        line=dict(color="#0d0d1a", width=2)
    ),
    text=text_labels,
    textinfo="text",
    textfont=dict(size=10, color="white"),
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Deaths: %{value:,}<br>"
        "Share: %{percent}<extra></extra>"
    ),
    name="State",
    pull=[0.05 if i < 3 else 0 for i in range(len(tw_all))]
))

# Center text
fig.add_annotation(
    text="<b>🏍️</b><br><b>Total</b><br><b>4,60,871</b><br><b>Deaths</b><br><b>(2015-2022)</b>",
    x=0.5, y=0.5,
    xref="paper", yref="paper",
    showarrow=False,
    font=dict(size=13, color="white"),
    align="center"
)

fig.update_layout(
    title=dict(
        text="<b>🏍️ Two Wheeler Heatwave</b><br>"
             "<sup>All States/UTs — Two Wheeler Deaths Share — India (2015-2022)</sup>",
        x=0.5,
        font=dict(size=18, color="white")
    ),
    paper_bgcolor="#0d0d1a",
    plot_bgcolor="#0d0d1a",
    font=dict(color="white"),
    showlegend=True,
    legend=dict(
        font=dict(color="white", size=9),
        bgcolor="rgba(0,0,0,0.3)",
        bordercolor="#e63946",
        borderwidth=1,
        x=1.02,
        y=0.5,
        tracegroupgap=1
    ),
    height=800,
    width=1200,
    margin=dict(t=130, b=50, l=50, r=250)
)

fig.show()
print("Smart Text Two Wheeler Donut rendered!")

In [ ]:
# Visualization 6 - The Fog Index

# Fog belt states
fog_states = ["Uttar Pradesh", "Punjab", "Haryana", "Delhi", "Bihar",
              "Rajasthan", "Madhya Pradesh", "Uttarakhand", "Himachal Pradesh",
              "Jammu and Kashmir", "Chandigarh", "West Bengal"]

# Winter months — Dec, Jan, Feb
winter_months = ["December", "January", "February"]
all_months = ["January", "February", "March", "April", "May", "June",
              "July", "August", "September", "October", "November", "December"]

# Night slots — 0000-0600 (peak fog hours)
fog_night_slots = [
    "0000 hrs to 0300 hrs (Night)",
    "0300 hrs to 0600 hrs (Night)"
]

# Calculate Fog Index per state per year
fog_month = month_df[month_df["State/UT"].isin(fog_states)].copy()
fog_time = time_df[time_df["State/UT"].isin(fog_states)].copy()

# Winter accidents
fog_month["Winter Accidents"] = fog_month[winter_months].sum(axis=1)
fog_month["Total Accidents"] = fog_month[all_months].sum(axis=1)
fog_month["Winter %"] = (fog_month["Winter Accidents"] / 
                          fog_month["Total Accidents"] * 100).round(1)

# Early morning night accidents
fog_time["Fog Night Accidents"] = fog_time[fog_night_slots].sum(axis=1)
fog_time["Total Time Accidents"] = fog_time[[
    "0000 hrs to 0300 hrs (Night)", "0300 hrs to 0600 hrs (Night)",
    "0600 hrs to 0900 hrs (Day)", "0900 hrs to 1200 hrs (Day)",
    "1200 hrs to 1500 hrs (Day)", "1500 hrs to 1800 hrs (Day)",
    "1800 hrs to 2100 hrs (Night)", "2100 hrs to 2400 hrs (Night)"
]].sum(axis=1)
fog_time["Fog Night %"] = (fog_time["Fog Night Accidents"] / 
                            fog_time["Total Time Accidents"] * 100).round(1)

# Merge
fog_merged = fog_month[["State/UT", "Year", "Winter Accidents", "Winter %"]].merge(
    fog_time[["State/UT", "Year", "Fog Night Accidents", "Fog Night %"]],
    on=["State/UT", "Year"]
)

# Fog Index = Winter % × Fog Night % / 100
fog_merged["Fog Index"] = (fog_merged["Winter %"] * 
                            fog_merged["Fog Night %"] / 100).round(2)

# State wise average
fog_state_avg = fog_merged.groupby("State/UT").agg(
    Fog_Index=("Fog Index", "mean"),
    Winter_Pct=("Winter %", "mean"),
    Fog_Night_Pct=("Fog Night %", "mean"),
    Winter_Accidents=("Winter Accidents", "sum")
).reset_index().sort_values("Fog_Index", ascending=False)

# Year wise trend
fog_year = fog_merged.groupby(["Year", "State/UT"])["Fog Index"].mean().reset_index()

# ============================================================
# PLOT
# ============================================================
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.45, 0.55],
    specs=[[{"type": "xy"}, {"type": "xy"}]],
)

# LEFT - Fog Index Bar Chart
colors_fog = [
    "#e63946" if x > fog_state_avg["Fog_Index"].mean() + fog_state_avg["Fog_Index"].std()
    else "#f4a261" if x > fog_state_avg["Fog_Index"].mean()
    else "#e9c46a"
    for x in fog_state_avg["Fog_Index"]
]

fig.add_trace(go.Bar(
    x=fog_state_avg["Fog_Index"],
    y=fog_state_avg["State/UT"],
    orientation="h",
    marker=dict(
        color=colors_fog,
        line=dict(color="#0d0d1a", width=0.5)
    ),
    text=[f"{v:.2f}" for v in fog_state_avg["Fog_Index"]],
    textposition="outside",
    textfont=dict(color="white", size=10),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Fog Index: %{x:.2f}<extra></extra>"
    ),
    showlegend=False
), row=1, col=1)

# Average line
avg_fog = fog_state_avg["Fog_Index"].mean()
fig.add_vline(
    x=avg_fog,
    line_dash="dash",
    line_color="white",
    line_width=1.5,
    row=1, col=1
)
fig.add_annotation(
    x=avg_fog, y=11.5,
    text=f"Avg: {avg_fog:.2f}",
    showarrow=False,
    font=dict(size=9, color="white"),
    bgcolor="rgba(0,0,0,0.5)",
    xref="x", yref="y"
)

# RIGHT - Year wise Trend
state_line_colors = [
    "#e63946", "#f4a261", "#e9c46a", "#2a9d8f",
    "#457b9d", "#1a1a4e", "#c1121f", "#ff6b35",
    "#8ecae6", "#a8dadc", "#06d6a0", "#118ab2"
]

for i, state in enumerate(fog_state_avg["State/UT"].tolist()):
    state_data_line = fog_year[fog_year["State/UT"] == state]
    fig.add_trace(go.Scatter(
        x=state_data_line["Year"],
        y=state_data_line["Fog Index"],
        mode="lines+markers",
        name=state,
        line=dict(
            color=state_line_colors[i % len(state_line_colors)],
            width=2
        ),
        marker=dict(size=6),
        hovertemplate=f"<b>{state}</b><br>Year: %{{x}}<br>Fog Index: %{{y:.2f}}<extra></extra>"
    ), row=1, col=2)

# Subtitles
fig.add_annotation(
    text="<b>❄️ Fog Index — State Ranking</b>",
    x=0.18, y=1.05,
    xref="paper", yref="paper",
    showarrow=False,
    font=dict(size=13, color="white")
)
fig.add_annotation(
    text="<b>📈 Fog Index — Year wise Trend</b>",
    x=0.75, y=1.05,
    xref="paper", yref="paper",
    showarrow=False,
    font=dict(size=13, color="white")
)

fig.update_layout(
    title=dict(
        text="<b>🌫️ The Fog Index — North India Winter Fog Belt (2015-2022)</b><br>"
             "<sup>Fog Index = Winter Accident % × Early Morning Night Accident %</sup>",
        x=0.5,
        font=dict(size=18, color="white")
    ),
    xaxis=dict(
        tickfont=dict(color="white", size=9),
        gridcolor="rgba(255,255,255,0.05)",
        title=dict(text="Fog Index", font=dict(color="white", size=10))
    ),
    yaxis=dict(
        tickfont=dict(color="white", size=9),
        gridcolor="rgba(255,255,255,0.05)"
    ),
    xaxis2=dict(
        tickfont=dict(color="white", size=9),
        gridcolor="rgba(255,255,255,0.05)",
        tickvals=list(range(2015, 2023)),
        title=dict(text="Year", font=dict(color="white", size=10))
    ),
    yaxis2=dict(
        tickfont=dict(color="white", size=9),
        gridcolor="rgba(255,255,255,0.05)",
        title=dict(text="Fog Index", font=dict(color="white", size=10))
    ),
    paper_bgcolor="#0d0d1a",
    plot_bgcolor="#0d0d1a",
    font=dict(color="white"),
    legend=dict(
        font=dict(color="white", size=8),
        bgcolor="rgba(0,0,0,0.3)",
        x=1.01,
        y=0.5
    ),
    height=650,
    width=1400,
    margin=dict(t=130, b=60, l=180, r=150)
)

fig.show()
print("Fog Index rendered!")

In [ ]:
# Define vehicle_cols first
vehicle_cols = ["Truck/Lorry/Mini Truck - Died", "Bus - Died", "SUV/Car/Jeep - Died",
                "Tractor - Died", "Three Wheeler/Auto Rickshaw - Died",
                "Two Wheeler - Died", "Other Motor Vehicles - Died"]

fog_night_slots = [
    "0000 hrs to 0300 hrs (Night)",
    "0300 hrs to 0600 hrs (Night)"
]

print("Variables defined!")

In [ ]:
# Visualization 7 - India Map - Fixed GeoJSON

import urllib.request
import json

# Try different GeoJSON source
geojson_url = "https://raw.githubusercontent.com/Subhash9325/GeoJson-Data-of-Indian-States/master/Indian_States"

try:
    with urllib.request.urlopen(geojson_url) as response:
        india_geojson = json.loads(response.read())
    print("GeoJSON loaded!")
    print(f"Features: {len(india_geojson['features'])}")
    # Check property keys
    print(f"Properties: {india_geojson['features'][0]['properties']}")
except Exception as e:
    print(f"Failed: {e}")

In [ ]:
# Visualization 7 - India Map - Local GeoJSON

import json

# Load local GeoJSON
with open("../data/maps/india_states.geojson", "r") as f:
    india_geojson = json.load(f)

print(f"GeoJSON loaded! Features: {len(india_geojson['features'])}")
print(f"Sample properties: {india_geojson['features'][0]['properties']}")

In [ ]:
# Check all NAME_1 values in GeoJSON
geojson_names = [f["properties"]["NAME_1"] for f in india_geojson["features"]]
print("GeoJSON state names:")
for name in sorted(geojson_names):
    print(f"  {name}")

In [ ]:
# Visualization 7 - India Map - FINAL

# Updated state mapping
state_name_map_v2 = {
    "Andaman and Nicobar Islands": "Andaman and Nicobar",
    "Andhra Pradesh": "Andhra Pradesh",
    "Arunachal Pradesh": "Arunachal Pradesh",
    "Assam": "Assam",
    "Bihar": "Bihar",
    "Chandigarh": "Chandigarh",
    "Chhattisgarh": "Chhattisgarh",
    "Dadra and Nagar Haveli and Daman and Diu": "Dadra and Nagar Haveli",
    "Delhi": "Delhi",
    "Goa": "Goa",
    "Gujarat": "Gujarat",
    "Haryana": "Haryana",
    "Himachal Pradesh": "Himachal Pradesh",
    "Jammu and Kashmir": "Jammu and Kashmir",
    "Jharkhand": "Jharkhand",
    "Karnataka": "Karnataka",
    "Kerala": "Kerala",
    "Lakshadweep": "Lakshadweep",
    "Madhya Pradesh": "Madhya Pradesh",
    "Maharashtra": "Maharashtra",
    "Manipur": "Manipur",
    "Meghalaya": "Meghalaya",
    "Mizoram": "Mizoram",
    "Nagaland": "Nagaland",
    "Odisha": "Orissa",
    "Puducherry": "Puducherry",
    "Punjab": "Punjab",
    "Rajasthan": "Rajasthan",
    "Sikkim": "Sikkim",
    "Tamil Nadu": "Tamil Nadu",
    "Telangana": "Andhra Pradesh",  # Telangana not in old GeoJSON
    "Tripura": "Tripura",
    "Uttar Pradesh": "Uttar Pradesh",
    "Uttarakhand": "Uttaranchal",
    "West Bengal": "West Bengal"
}

# Prepare map data
map_deaths = deaths_df.groupby("State/UT")["Total Accidental Deaths"].mean().reset_index()
map_deaths.columns = ["State/UT", "Avg Deaths"]
map_deaths["Map State"] = map_deaths["State/UT"].map(state_name_map_v2)

# Two Wheeler %
tw_map = transport_df.copy()
tw_map["Total Vehicle"] = tw_map[vehicle_cols].sum(axis=1)
tw_map["TW %"] = (tw_map["Two Wheeler - Died"] / tw_map["Total Vehicle"] * 100)
tw_map = tw_map.groupby("State/UT")["TW %"].mean().reset_index()
map_deaths = map_deaths.merge(tw_map, on="State/UT")

# Night %
night_map = time_df.copy()
night_map["Night"] = night_map[[
    "0000 hrs to 0300 hrs (Night)", "0300 hrs to 0600 hrs (Night)",
    "1800 hrs to 2100 hrs (Night)", "2100 hrs to 2400 hrs (Night)"
]].sum(axis=1)
night_map["Total"] = night_map[[
    "0000 hrs to 0300 hrs (Night)", "0300 hrs to 0600 hrs (Night)",
    "0600 hrs to 0900 hrs (Day)", "0900 hrs to 1200 hrs (Day)",
    "1200 hrs to 1500 hrs (Day)", "1500 hrs to 1800 hrs (Day)",
    "1800 hrs to 2100 hrs (Night)", "2100 hrs to 2400 hrs (Night)"
]].sum(axis=1)
night_map["Night %"] = (night_map["Night"] / night_map["Total"] * 100)
night_map = night_map.groupby("State/UT")["Night %"].mean().reset_index()
map_deaths = map_deaths.merge(night_map, on="State/UT")

# Deaths per Lakh
map_deaths["Population"] = map_deaths["State/UT"].map(population_data)
map_deaths["Deaths per Lakh"] = (map_deaths["Avg Deaths"] /
                                  map_deaths["Population"]).round(1)

# Merge Telangana + AP
map_deaths_grouped = map_deaths.groupby("Map State").agg({
    "Avg Deaths": "sum",
    "TW %": "mean",
    "Night %": "mean",
    "Deaths per Lakh": "mean"
}).reset_index()

# Metrics config
metrics_map = {
    "Avg Deaths": "Average Annual Deaths",
    "Deaths per Lakh": "Deaths per Lakh Population",
    "TW %": "Two Wheeler % of Deaths",
    "Night %": "Night Accidents %"
}

colorscales = {
    "Avg Deaths": [[0, "#0d1b2a"], [0.5, "#e63946"], [1, "#ff9f1c"]],
    "Deaths per Lakh": [[0, "#0d1b2a"], [0.5, "#c77dff"], [1, "#ff9f1c"]],
    "TW %": [[0, "#0d1b2a"], [0.5, "#f4a261"], [1, "#e63946"]],
    "Night %": [[0, "#0d1b2a"], [0.5, "#457b9d"], [1, "#e63946"]]
}

fig = go.Figure()

for i, (col, label) in enumerate(metrics_map.items()):
    fig.add_trace(go.Choropleth(
        geojson=india_geojson,
        locations=map_deaths_grouped["Map State"],
        z=map_deaths_grouped[col],
        featureidkey="properties.NAME_1",
        colorscale=colorscales[col],
        showscale=True,
        colorbar=dict(
            title=dict(text=label, font=dict(color="white", size=10)),
            tickfont=dict(color="white"),
            len=0.6,
            thickness=15
        ),
        hovertemplate=(
            "<b>%{location}</b><br>" +
            f"{label}: %{{z:.1f}}<extra></extra>"
        ),
        visible=(i == 0),
        name=label,
        marker=dict(line=dict(color="rgba(255,255,255,0.3)", width=0.5))
    ))

# Dropdown buttons
map_buttons = []
for i, (col, label) in enumerate(metrics_map.items()):
    visibility = [False] * len(metrics_map)
    visibility[i] = True
    map_buttons.append(dict(
        label=label,
        method="update",
        args=[
            {"visible": visibility},
            {"title": {"text": f"<b>\U0001f5fa\ufe0f India Road Accidents Map</b><br>"
                       f"<sup>{label} — 2015-2022</sup>"}}
        ]
    ))

fig.update_layout(
    title=dict(
        text="<b>\U0001f5fa\ufe0f India Road Accidents Map</b><br>"
             "<sup>Average Annual Deaths — 2015-2022 | Use dropdown to switch metrics</sup>",
        x=0.5,
        font=dict(size=18, color="white")
    ),
    updatemenus=[dict(
        buttons=map_buttons,
        direction="down",
        showactive=True,
        x=0.02,
        xanchor="left",
        y=1.12,
        yanchor="top",
        bgcolor="#1a1a2e",
        bordercolor="#e63946",
        font=dict(color="white", size=11)
    )],
    geo=dict(
        scope="asia",
        center=dict(lat=23, lon=80),
        projection_scale=4.5,
        showland=True,
        landcolor="#1a1a2e",
        showocean=True,
        oceancolor="#0d0d1a",
        showcountries=True,
        countrycolor="rgba(255,255,255,0.2)",
        showframe=False,
        bgcolor="#0d0d1a",
        showcoastlines=True,
        coastlinecolor="rgba(255,255,255,0.2)"
    ),
    paper_bgcolor="#0d0d1a",
    font=dict(color="white"),
    height=800,
    width=1000,
    margin=dict(t=130, b=30, l=30, r=30)
)

fig.show()
print("India Map rendered!")